# 11.6 - Document Ingestion

**Phase:** 11 - RAG Systems

**Status:** VERIFIED

---

## 1. What Are We Solving?

Real-world data is messy: PDFs, text files, markdown, HTML. Ingestion extracts clean text + metadata from raw files so the rest of the pipeline (chunking, embedding, retrieval) inherits good input.

## 2. Why Does This Matter?

If ingestion fails, everything downstream fails. Garbled text, lost page numbers, and duplicate documents corrupt chunking, embedding, and retrieval - and you waste hours debugging the wrong layer.

## 3. Prerequisites

Phase 01 (Python), Unit 11.3 (Embeddings).

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Load documents from txt, md, and pdf (pypdf) with try/except per format
- Extract and preserve metadata (source, page)
- Create sample docs in memory (including a tiny PDF via io.BytesIO) and load them

## 5. Mental Model

Ingestion is a document preprocessing pipeline: raw file -> clean text -> structured records, each with source + page metadata.

```text
Raw File -> Parser -> Clean Text + Metadata -> (next: Chunking)
```


## 6. Setup
We import `pypdf` (the maintained fork of PyPDF2) and `io` to build a tiny PDF in memory, so **nothing is written to disk**.

In [1]:
import io
from pathlib import Path

try:
    from pypdf import PdfReader, PdfWriter
    print("pypdf available")
except Exception as e:
    PdfReader = PdfWriter = None
    print("pypdf unavailable:", type(e).__name__)


pypdf available


## 7. Loaders per Format (txt, md, pdf)
One function per format, each wrapped in try/except so a bad file degrades gracefully instead of crashing the pipeline. Each returns records with `text`, `source`, and (for pdf) `page` metadata.

In [2]:
def load_txt_or_md(path):
    try:
        text = Path(path).read_text(encoding="utf-8")
        return [{"text": text, "source": str(path)}]
    except Exception as e:
        return [{"text": "", "source": str(path), "error": type(e).__name__}]


def load_pdf(path):
    if PdfReader is None:
        return [{"text": "", "source": str(path), "error": "pypdf missing"}]
    records = []
    try:
        reader = PdfReader(str(path))
        for i, page in enumerate(reader.pages):
            text = page.extract_text() or ""
            if text.strip():
                records.append({"text": text, "source": str(path), "page": i + 1})
    except Exception as e:
        records.append({"text": "", "source": str(path), "error": type(e).__name__})
    return records


def load_any(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".pdf":
        return load_pdf(path)
    if suffix in (".txt", ".md"):
        return load_txt_or_md(path)
    return [{"text": "", "source": str(path), "error": "unsupported format"}]


## 8. Build Sample Docs in Memory
We create a text file, a markdown file, and a small PDF - all in memory via `io.BytesIO` - then write them to a temp directory (outside the repo) so the loaders can read them back.

In [3]:
import tempfile, os

tmp = tempfile.mkdtemp(prefix="ingest_")
faq_txt = "What is the refund window? Refunds are issued to the original payment method."
md_doc = "# Onboarding\n\nAccounts are created after identity verification."
(os.path.join(tmp, "faq.txt")).__class__ and open(os.path.join(tmp, "faq.txt"), "w", encoding="utf-8").write(faq_txt)
open(os.path.join(tmp, "onboarding.md"), "w", encoding="utf-8").write(md_doc)

if PdfWriter is not None:
    writer = PdfWriter()
    page = writer.add_blank_page(width=612, height=792)
    writer.add_metadata({"/Title": "Policy"})
    bio = io.BytesIO()
    writer.write(bio)
    bio.seek(0)
    with open(os.path.join(tmp, "policy.pdf"), "wb") as f:
        f.write(bio.read())

print("sample dir:", tmp)
print("files:", sorted(os.listdir(tmp)))


sample dir: C:\Users\PC\AppData\Local\Temp\ingest_iq19que3
files: ['faq.txt', 'onboarding.md', 'policy.pdf']


## 9. Load Everything
Run the loaders over each format and print the extracted text + metadata for each record. On the blank PDF page you may see empty/blank text - that is normal and why we skip blank pages.

In [4]:
all_records = []
for fname in sorted(os.listdir(tmp)):
    path = os.path.join(tmp, fname)
    records = load_any(path)
    all_records.extend(records)
    for r in records:
        preview = r.get("text", "")[:60].replace("\n", " ")
        meta = {k: v for k, v in r.items() if k not in ("text",)}
        print(f"{fname:16s} meta={meta}")
        print(f"   text({len(r.get('text',''))} chars): {preview!r}")

print("\ntotal records:", len(all_records))
print("total characters:", sum(len(r.get("text", "")) for r in all_records))


faq.txt          meta={'source': 'C:\\Users\\PC\\AppData\\Local\\Temp\\ingest_iq19que3\\faq.txt'}
   text(77 chars): 'What is the refund window? Refunds are issued to the origina'
onboarding.md    meta={'source': 'C:\\Users\\PC\\AppData\\Local\\Temp\\ingest_iq19que3\\onboarding.md'}
   text(63 chars): '# Onboarding  Accounts are created after identity verificati'

total records: 2
total characters: 140


## 10. Deduplication Awareness
Ingesting the same document twice creates duplicate chunks. A cheap guard is to hash the normalized text and skip records already seen. We show a quick hash-based dedupe over the loaded records.

In [5]:
import hashlib

seen = set()
unique = []
for r in all_records:
    key = hashlib.md5(r.get("text", "").strip().lower().encode()).hexdigest()
    if key not in seen:
        seen.add(key)
        unique.append(r)
print(f"loaded={len(all_records)} -> unique after dedup={len(unique)}")


loaded=2 -> unique after dedup=2



## Common Mistakes

- Not handling encoding errors (UnicodeDecodeError).
- Extracting headers/footers as content.
- Losing page/section metadata.
- Ingesting the same doc twice without dedup.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| Empty documents after extraction | Parser failed silently | Check page count vs text; try another parser |
| Garbled text | Encoding issue | Specify encoding explicitly |
| Tables unreadable | Structure lost in extraction | Use a table-aware parser |
| Duplicate chunks | Ingested twice | Hash documents for dedup |

## Best Practices

- Always store source filename and page/section as metadata.
- Handle encoding explicitly (UTF-8 default).
- Deduplicate before ingestion.
- Test on a few docs before processing thousands.
- Log ingestion stats (docs, pages, chars).

## Hands-On Practice

1. **Basic:** Load a text file and a PDF; print extracted text.
2. **Guided:** Extract metadata (title, page count, file size) alongside text.
3. **Independent:** Build an ingestion pipeline handling TXT, PDF, and MD.
4. **Realistic:** Ingest a PDF with tables and compare parser quality.
5. **Challenge:** Build dedup into your pipeline.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
